In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill
import torch
import pickle



In [4]:
from sentence_transformers import CrossEncoder,InputExample

from torch.utils.data import DataLoader

from src.metric import model_evaluation


c:\Users\ankes\.conda\envs\resume-job-analyzer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
with open('../data/cleaned/train_df.pkl','rb') as f:
    train_df=pickle.load(f)
    
with open('../data/cleaned/val_df.pkl','rb') as f:
    val_df=pickle.load(f)
        
with open('../data/cleaned/test_df.pkl','rb') as f:
    test_df=pickle.load(f)
    


In [8]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [ ]:
label_to_score = {0: 0.0, 1: 0.5, 2: 1.0}


cross_train_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

print(f"total training example:{len(cross_train_examples)}")

total training example:5073


In [ ]:
cross_train_dataloader=DataLoader(cross_train_examples,shuffle=True,batch_size=64)

In [12]:
# cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=device, max_length=384)
cross_encoder_model = CrossEncoder("../models/cross_encoder")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1296.22it/s]


In [ ]:
epochs=4
best_score=float('-inf')
min_delta=0.01
patience=2
count=0

for epoch in range(1,epochs+1):
    print(f"Epoch: {epoch}----------")

    cross_encoder_model.fit(train_dataloader=cross_train_dataloader,epochs=1,
                             warmup_steps=int(len(cross_train_dataloader) * epochs * 0.1),
                            show_progress_bar=True)
    
    val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

    scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)
    
    metrics=model_evaluation(scores,val_df,'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])    

    final_score =(0.6*metrics['ndcg_val'] +
                   0.3*metrics['map_score'] +0.1*metrics['mrr_score'])
    
    if final_score>best_score+0.01:
        best_score=final_score
        cross_encoder_model.save("../models/baseline/cross_encoder")
        count=0
    else:
        count+=1
        
    if count==patience:
        print("Early Stopping")
        break
    


val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)


In [ ]:
metrics=model_evaluation(scores,val_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

	Inference Phase


Batches: 100%|██████████| 37/37 [01:26<00:00,  2.35s/it]


Spearman: 0.4011489733791305
Top-3 Accuracy: 1.0
NDCG: 0.6855487316150893
MRR: 0.8140873015873016
MAP: 0.755965025666324


{'spearman_score': np.float64(0.4011489733791305),
 'topk_score': np.float64(1.0),
 'ndcg_val': np.float64(0.6855487316150893),
 'mrr_score': np.float64(0.8140873015873016),
 'map_score': np.float64(0.755965025666324)}

In [ ]:
ranked_result=[]
eval_df=val_df.copy()
eval_df['score']=scores

In [ ]:
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")

In [ ]:
for jd,group in eval_df.groupby('job_description_text'):
    ranked_group=group.sort_values("score",ascending=False)
    ranked_result.append(ranked_group)

final_rank_df=pd.concat(ranked_result)

In [16]:
i=0
for jd,group in final_rank_df.groupby('job_description_text'):
    if(len(group)>2 and len(group)<10):
        print("Job Description:\n",jd[:300])
        print(group[['label','score']])
        i+=1
        if i==3:
            break

Job Description:
 About Chamberlain Group:
Chamberlain Group is a global leader in access solutions. Our leading brands like LiftMaster, Chamberlain, Merlin and Grifco are found in millions of homes and commercial applications across the globe. Our innovative products powered by the myQ digital ecosystem provide cust
      label     score
518       0  0.110396
1326      0  0.025011
1349      0 -0.075225
442       0 -0.169593
3022      0 -0.178342
939       0 -0.539382
3684      1 -0.664484
1833      0 -1.025583
Job Description:
 About Hallgate Management: Hallgate Management is a property management company with a strong commitment to providing exceptional service to our clients and residents. We pride ourselves on our dedication to excellence, integrity, and continuous growth.
Position Overview: We are seeking a detail-ori
      label     score
401       0 -0.404804
1309      0 -0.459419
300       0 -0.884121
1129      0 -1.068836
894       0 -1.844438
Job Description:
 About Us Skadd

In [ ]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=64,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])


Batches: 100%|██████████| 55/55 [02:34<00:00,  2.81s/it]


Spearman: 0.261209752236857
Top-3 Accuracy: 0.9285714285714286
NDCG: 0.6128210653499944
MRR: 0.788544106167057
MAP: 0.7115900311976464


{'spearman_score': np.float64(0.261209752236857),
 'topk_score': np.float64(0.9285714285714286),
 'ndcg_val': np.float64(0.6128210653499944),
 'mrr_score': np.float64(0.788544106167057),
 'map_score': np.float64(0.7115900311976464)}